In [1]:
!pip install ucimlrepo pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings('ignore')

print("Fetching dataset directly from UCI Machine Learning Repository...")
# 1. Fetch the dataset automatically
ckd_data = fetch_ucirepo(id=336)

# Extract features and targets into pandas DataFrames
X_raw = ckd_data.data.features
y_raw = ckd_data.data.targets

# Combine into a single DataFrame to process
df = pd.concat([X_raw, y_raw], axis=1)

# Rename the target column (UCI names it 'class', we want 'classification')
df = df.rename(columns={'class': 'classification'})

# 2. Data Preprocessing
print("Preprocessing data...")
# Clean up common dirty data markers found in this specific dataset
df = df.replace(to_replace={'\t?': np.nan, '?': np.nan, '\t': np.nan})

# Define numerical and categorical columns
# (Note: UCI API uses 'wbcc' and 'rbcc' instead of 'wc' and 'rc')
numerical_cols = ['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']
categorical_cols = ['sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']

# Convert numerical columns to float
for col in numerical_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Impute missing values
# Numerical: replace missing with mean
num_imputer = SimpleImputer(strategy='mean')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Categorical: replace missing with mode (most frequent)
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Encode Categorical Variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Encode the Target Variable ('classification')
# Clean typos in the target column (e.g., 'ckd\t') and map to 1 (normal) and 0 (sickness)
df['classification'] = df['classification'].replace({'ckd\t': 'ckd', 'notckd': 'notckd'})
df['classification'] = df['classification'].map({'notckd': 1, 'ckd': 0})

# Drop any rows where the target is still NaN
df = df.dropna(subset=['classification'])

X = df.drop('classification', axis=1)
y = df['classification']

# 3. Data Splitting & Scaling
# 80% training and 20% testing split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Model Initialization
lr = LogisticRegression(random_state=42, max_iter=1000)
dt = DecisionTreeClassifier(random_state=42)
svm = SVC(kernel='linear', probability=True, random_state=42)

# Bagging Ensembles
bagging_lr = BaggingClassifier(estimator=lr, n_estimators=10, random_state=42)
bagging_dt = BaggingClassifier(estimator=dt, n_estimators=10, random_state=42)
bagging_svm = BaggingClassifier(estimator=svm, n_estimators=10, random_state=42)

models = {
    "Logistic Regression (LR)": lr,
    "Decision Tree (DT)": dt,
    "Support Vector Machine (SVM)": svm,
    "Bagging + LR": bagging_lr,
    "Bagging + DT": bagging_dt,
    "Bagging + SVM": bagging_svm
}

# 5. Training and Evaluation
results = []

print("Training models and performing Cross Validation...")
cv = KFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    # Fit the model on the full training set
    model.fit(X_train_scaled, y_train)

    # Predict on the 20% testing set
    y_pred = model.predict(X_test_scaled)

    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    results.append({
        "Model": name,
        "Accuracy (%)": round(acc, 2),
        "Precision": round(prec, 2),
        "Recall": round(rec, 2),
        "F-score": round(f1, 2)
    })

# 6. Generate the Comparison Table
results_df = pd.DataFrame(results)

print("\n" + "="*70)
print(" COMPARISON TABLE (Replicating Table 5 Metrics)")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

Fetching dataset directly from UCI Machine Learning Repository...
Preprocessing data...
Training models and performing Cross Validation...

 COMPARISON TABLE (Replicating Table 5 Metrics)
                       Model  Accuracy (%)  Precision  Recall  F-score
    Logistic Regression (LR)        100.00       1.00    1.00     1.00
          Decision Tree (DT)         98.75       0.99    0.99     0.99
Support Vector Machine (SVM)        100.00       1.00    1.00     1.00
                Bagging + LR        100.00       1.00    1.00     1.00
                Bagging + DT         98.75       0.99    0.99     0.99
               Bagging + SVM        100.00       1.00    1.00     1.00
